In [1]:
from sklearn.datasets import fetch_20newsgroups

newsgroups = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
texts = newsgroups.data

print("Nombre de documents :", len(texts))
print("\nExemple de texte :\n", texts[0][:500])

Nombre de documents : 11314

Exemple de texte :
 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000, stop_words='english', min_df=5)
X_tfidf = vectorizer.fit_transform(texts)

print("Shape de la matrice TF-IDF :", X_tfidf.shape)

Shape de la matrice TF-IDF : (11314, 1000)


In [10]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

silhouette_scores =[]
k_range=range(2,10)
for k in k_range:
    kmeans= KMeans(n_clusters=k, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_tfidf)
    score = silhouette_score(X_tfidf, clusters)
    silhouette_scores.append(score)
    print(f"k={k}: slihouette_scores={score:.4f}")

k=2: slihouette_scores=0.0106
k=3: slihouette_scores=0.0107
k=4: slihouette_scores=0.0099
k=5: slihouette_scores=0.0104
k=6: slihouette_scores=0.0117
k=7: slihouette_scores=0.0127
k=8: slihouette_scores=0.0127
k=9: slihouette_scores=0.0147


In [11]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=100, random_state=42)
X_reduced = svd.fit_transform(X_tfidf)

print("Nouvelle shape :", X_reduced.shape)
print("Variance totale expliquée :", svd.explained_variance_ratio_.sum())

Nouvelle shape : (11314, 100)
Variance totale expliquée : 0.2853138444528491


In [ ]:
silhouette_scores_svd = []
K_range = range(2, 15)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_reduced)
    score = silhouette_score(X_reduced, clusters)
    silhouette_scores_svd.append(score)
    print(f"k={k} : silhouette score = {score:.4f}")

k=2 : silhouette score = 0.0324
k=3 : silhouette score = 0.0359
k=4 : silhouette score = 0.0615
k=5 : silhouette score = 0.0383
k=6 : silhouette score = 0.0330
k=7 : silhouette score = 0.0364
k=8 : silhouette score = 0.0397
k=9 : silhouette score = 0.0451
k=10 : silhouette score = 0.0416
k=11 : silhouette score = 0.0386


In [15]:
# Entraînement final avec k=4
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters_final = kmeans_final.fit_predict(X_reduced)

# Ramener les centres de clusters dans l'espace TF-IDF original
centers_original_space = svd.inverse_transform(kmeans_final.cluster_centers_)

# Récupérer le vocabulaire (les 1000 mots)
terms = vectorizer.get_feature_names_out()

# Pour chaque cluster, afficher les 10 mots les plus représentatifs
for i in range(4):
    top_indices = centers_original_space[i].argsort()[::-1][:10]
    top_words = [terms[idx] for idx in top_indices]
    print(f"\n=== Cluster {i} ===")
    print(", ".join(top_words))


=== Cluster 0 ===
just, don, people, like, think, know, good, time, new, right

=== Cluster 1 ===
game, team, year, games, players, season, play, hockey, win, think

=== Cluster 2 ===
god, jesus, people, bible, believe, christ, christian, christians, faith, say

=== Cluster 3 ===
thanks, windows, use, does, drive, know, card, file, mail, problem
